# Charting the Grind

A project for the **Information Visualization course 2025/26** about understanding **personal study habits** through the visual exploration of a manually self-tracked study log, collected over roughly the last month of study sessions.

I started keeping this log because I wanted a clearer picture of what my study life actually looks like, beyond the vague feeling of "I studied a lot" or "this afternoon was not very productive". By looking at my sessions visually, I want to notice patterns I would probably miss otherwise: which topics take most of my time, when I seem to work better, which places help me focus and which kinds of activities feel more or less productive.

## Preparing the data

Before creating the visualizations, the raw study log needs to be cleaned and enriched. The CSV already contains the basic information about each study session, but some columns are still difficult to analyze directly. Primarily, start and finish times are written as text and the actual duration of each session is not explicitly available.  
In this phase, I **validate the dataset structure** and prepare it so that it can be used more easily for charts and interpretation.

First, I prepare the environment by importing the **libraries** I need and declaring all the **constants** used throughout the notebook.


In [1]:
from pathlib import Path
from IPython.display import display
import pandas as pd
import plotly.express as px

In [2]:
# the input data csv file path
DATA_CSV = Path("data/study_sessions.csv")

# the output directory for the generated charts images
CHARTS_DIR = Path("charts")

# the output directory for the generated chart for the website
WEBSITE_DIR = Path("website")

# the columns that should be treated as categorical
CATEGORICAL_COLS = ["topic", "activity_type", "location"]

# the mapping of time of day bins for categorization
TIME_OF_DAY_CUTS = {
    "bins": [8, 13.5, 18.5, 24],
    "labels": ["morning", "afternoon", "evening"],
}

Then, I read the data from the CSV file into a **dataframe** and check its structure to make sure it contains all the expected columns, that they are in the right format, and that there are **no missing values** or other issues that could affect the analysis.

In [3]:
df = pd.read_csv(DATA_CSV)

In [4]:
# check the structure of the dataframe, especially the data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   date                100 non-null    str  
 1   start               100 non-null    str  
 2   finish              100 non-null    str  
 3   topic               100 non-null    str  
 4   activity_type       100 non-null    str  
 5   productivity_score  100 non-null    int64
 6   location            100 non-null    str  
 7   description         100 non-null    str  
dtypes: int64(1), str(7)
memory usage: 6.4 KB


I now need to process the data a little to make it more suitable for analysis.

First, let's transform the **start** and **finish** times from text to datetime format and calculate the **duration** of each session in hours and minutes. This will allow us to analyze how long each session lasted and to create time-based visualizations later on.

In [5]:
# combine date + time into datetime columns
df["start_dt"] = pd.to_datetime(df["date"] + " " + df["start"], dayfirst=True)
df["finish_dt"] = pd.to_datetime(df["date"] + " " + df["finish"], dayfirst=True)

# calculate duration as a time delta
df["duration"] = df["finish_dt"] - df["start_dt"]

# convert duration to hours as a float
df["duration_hours"] = df["duration"].dt.total_seconds() / 3600

# convert duration to minutes as an integer
df["duration_minutes"] = (df["duration"].dt.total_seconds() / 60).astype(int)

# display the start, finish and duration columns to check the calculations
display(df[["start_dt", "finish_dt", "duration_hours", "duration_minutes"]])

,start_dt,finish_dt,duration_hours,duration_minutes
0,2026-04-13 10:00:00,2026-04-13 12:30:00,2.5,150
1,2026-04-14 09:30:00,2026-04-14 12:00:00,2.5,150
2,2026-04-14 15:30:00,2026-04-14 17:00:00,1.5,90
3,2026-04-14 17:00:00,2026-04-14 17:30:00,0.5,30
4,2026-04-14 18:30:00,2026-04-14 21:00:00,2.5,150
...,...,...,...,...
95,2026-05-19 21:00:00,2026-05-19 23:30:00,2.5,150
96,2026-05-20 09:00:00,2026-05-20 11:00:00,2.0,120
97,2026-05-20 11:00:00,2026-05-20 12:00:00,1.0,60
98,2026-05-20 16:00:00,2026-05-20 18:00:00,2.0,120


Then, let's infer the **time of the day** for each session based on the start time. This will help us understand when I tend to study and whether there are any patterns in my productivity throughout the day.

In [6]:
# infer time of day from the start time using the defined bins and labels
df["time_of_day"] = pd.cut(
    df["start_dt"].dt.hour + df["start_dt"].dt.minute / 60,
    **TIME_OF_DAY_CUTS,
    right=False,
)

# print the start, finish and time of day columns to check the categorization
display(df[["start_dt", "finish_dt", "time_of_day"]])

,start_dt,finish_dt,time_of_day
0,2026-04-13 10:00:00,2026-04-13 12:30:00,morning
1,2026-04-14 09:30:00,2026-04-14 12:00:00,morning
2,2026-04-14 15:30:00,2026-04-14 17:00:00,afternoon
3,2026-04-14 17:00:00,2026-04-14 17:30:00,afternoon
4,2026-04-14 18:30:00,2026-04-14 21:00:00,evening
...,...,...,...
95,2026-05-19 21:00:00,2026-05-19 23:30:00,evening
96,2026-05-20 09:00:00,2026-05-20 11:00:00,morning
97,2026-05-20 11:00:00,2026-05-20 12:00:00,morning
98,2026-05-20 16:00:00,2026-05-20 18:00:00,afternoon


In [7]:
# create a mapping of location short names to more descriptive names
location_names = {
    "home": "Home",
    "bigiavi": "Bigiavi",
    "paleotti": "Palazzo Paleotti",
    "32": "Zamboni 32",
    "34": "Zamboni 34",
    "36": "Zamboni 36",
}

# replace the location values in the dataframe using the mapping
df["location"] = df["location"].replace(location_names)

Finally, let's convert some of the columns that I know should be categorical to the appropriate data type for easier analysis.

In [8]:
# convert columns marked as categorical to category data type
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype("category")

# print the unique values in categorical columns
for col in CATEGORICAL_COLS:
    unique_values = df[col].unique().tolist()
    unique_values.sort()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'topic': ['Digital Scholarly Editing', 'Freelance Work', 'Information Visualization', 'Knowledge Representation', 'Open Science', 'Semantic Digital Libraries']
Unique values in 'activity_type': ['brainstorming', 'programming', 'reading', 'reviewing', 'writing']
Unique values in 'location': ['Bigiavi', 'Home', 'Palazzo Paleotti', 'Zamboni 32', 'Zamboni 34', 'Zamboni 36']


In [9]:
# final preflight check of the dataframe structure after all transformations
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   date                100 non-null    str            
 1   start               100 non-null    str            
 2   finish              100 non-null    str            
 3   topic               100 non-null    category       
 4   activity_type       100 non-null    category       
 5   productivity_score  100 non-null    int64          
 6   location            100 non-null    category       
 7   description         100 non-null    str            
 8   start_dt            100 non-null    datetime64[us] 
 9   finish_dt           100 non-null    datetime64[us] 
 10  duration            100 non-null    timedelta64[us]
 11  duration_hours      100 non-null    float64        
 12  duration_minutes    100 non-null    int64          
 13  time_of_day         100 non-null    category   

## Visualizing the data

After preparing the dataframe and adding the columns needed for analysis, I can now start **turning the data into visualizations**.  
In this phase, the goal is to move from the raw list of study sessions to charts that make my **habits understandable** at a glance, **highlight meaningful patterns** and possibly help me reflect on **how to organize future study** time more consciously.


### Visualization 1: Total study time per Topic

The most obvious information I want to visualize is how much time I have spent on **each topic** compared to the others, and whether this confirms my **intuitive feeling** of where my study time has gone. The data is not complete: some topics had already been started, or were almost finished, when I began keeping the log, while others will continue after the project has been delivered. Still, this visualization is useful as a quick **vibe check**, because it gives me an immediate overview of the relative weight of each topic inside the recorded period.

To visualize this, I will **group the sessions** by `topic` and sum the `duration_hours` column, which was calculated from the `start` and `finish` times during the data preparation phase. The result will be a simple but effective **bar chart**, where each bar represents a topic and its total recorded study time. This makes it easy to compare topics at a glance and see which ones absorbed most of my attention during the logging period.

In [10]:
# group data by topic and calculate total study time using the duration_hours, then sort in ascending order
topic_hours = (
    df.groupby("topic", as_index=False)["duration_hours"]
    .sum()
    .sort_values("duration_hours", ascending=True)
)

# display the new dataframe for a quick check of the results
display(topic_hours)

,topic,duration_hours
5,Semantic Digital Libraries,8.50
0,Digital Scholarly Editing,12.50
1,Freelance Work,17.75
2,Information Visualization,24.25
3,Knowledge Representation,58.50
4,Open Science,73.25


In [11]:
# create a horizontal bar chart and customize it with titles, labels and colors
fig = px.bar(
    topic_hours,
    x="duration_hours",
    y="topic",
    orientation="h",
    title="Total study time per Topic",
    labels={
        "duration_hours": "Total study time (hours)",
        "topic": "Topic"
    },
    text="duration_hours",
    color_discrete_sequence=["#4C78A8"]
)

# show the total hours as text labels alongside the bars with a single decimal place
fig.update_traces(
    texttemplate="%{text:.1f} h",
    textposition="outside"
)

# compute the maximum hours value for x-axis padding for better visualization of the text labels
max_hours = topic_hours["duration_hours"].max()

# customize the layout of the figure for better aesthetics and readability
fig.update_layout(
    height=600,
    showlegend=False,
    title_font=dict(size=25),
    title=dict(
        text=(
            "A Geography of Studies<br>"
            "<sup>Total study time per topic</sup>"
        ),
        x=0.07,
        xanchor="left"
    ),
    xaxis_title="Total study time (hours)",
    yaxis_title="Topic",
    margin=dict(l=220, r=130, t=100, b=140),
    xaxis=dict(
        range=[0, max_hours * 1.15]
    ),
    yaxis=dict(
        title=dict(
            text="Topic",
            standoff=25
        )
    ),
    annotations=[
        dict(
            text="<b>Figure 1.</b> This chart compares the total amount of recorded study time for each topic: "
                 "it gives a quick overview of which subjects took up "
                 "most of my<br>attention during the logging period.",
            x=-0.1,
            y=-0.3,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=11),
        )
    ]
)

# save the figure as a static high-resolution image to the charts directory
fig.write_image(
    CHARTS_DIR / "total_study_time_per_topic.jpg",
    width=1200,
    height=600,
    scale=2
)

# display the figure in the notebook
fig.show()

The chart confirmed most of my expectations: the topics that felt more present in my daily routine are also the ones with the highest number of recorded study hours.

- **Open Science** requires most of my time and attention because I need to write complex Python code to manipulate huge amount of data for the exam project. I've written and rewritten the code several times, and I still have a lot of work to do, so it's not surprising that this topic takes up most of my study time.

- **Knowledge Representation** is the second topic in terms of hours, and this also makes sense because it was the exam I started working on when I began keeping the log. Moreover, while I passed the exam more than a week before the end of the logging period, I still spent some time on it to prepare a new release of the ontology we worked on in order to publish it on Zenodo.

- **Information Visualization** is the third topic and it's in the right place. It doesn't have the frustrating complexity of Open Science, or the obsessive perfectionism of Knowledge Representation, but it skyrocketed in the last week of the logging period because I had to prepare the project for submission and I wanted to make it as good as possible.

- **Freelance Work** and **Semantic Digital Libraries** are also quite where I expected. I stopped doing freelance work when the exam period started (and also because I didn't like the company I was working for anymore 🤭), while the SDL exam was a few days after the logging started.

- **Digital Scholarly Editing** however is surprisingly low in terms of hours, since I have the exam the same day as Information Visualization and I expected to have spent more time on it. So I will probably fail the exam 😎.

---

### Visualization 2: Total study time per Activity

After looking at study time by **topic**, I also want to understand how much time I have spent on **different kinds of activities**. This is slightly different from the previous visualization: a topic tells me **what** I was studying, while the activity type tells me **how** I was studying. For example, two sessions may belong to the same topic, but one could be reading, another writing, another coding, reviewing, or brainstorming. This helps me check whether my study routine is multi-faceted or mostly focused on a single type of activity.

To visualize this, I will **group the sessions** by `activity_type` and sum the `duration_hours` column. This time, I will use a **donut chart** to show the proportion of total study time dedicated to each activity. I chose this kind of visualization because I want to see what my study time is actually made of: instead of only asking how many hours I spent, I want to understand how that time is divided between different activities, and what role each activity plays in the overall shape of my study routine.

In [12]:
# group data by activity_type and calculate total time using the duration_hours, then sort in descending order
activity_hours = (
    df.groupby("activity_type", as_index=False)["duration_hours"]
    .sum()
    .sort_values("duration_hours", ascending=False)
)

# display the new dataframe for a quick check of the results
display(activity_hours)

,activity_type,duration_hours
1,programming,102.50
2,reading,38.50
4,writing,28.25
3,reviewing,14.50
0,brainstorming,11.00


In [13]:
# create a donut chart and customize it with titles, labels and color palette
fig = px.pie(
    activity_hours,
    names="activity_type",
    values="duration_hours",
    hole=0.45,
    title="Total study time per Activity",
    labels={
        "activity_type": "Activity type",
        "duration_hours": "Total study time (hours)"
    },
    color="activity_type",
    color_discrete_map={
        "programming": "#1982C4",
        "reading": "#8AC926",
        "writing": "#FF595E",
        "reviewing": "#FFCA3A",
        "brainstorming": "#6A4C93",
    }
)

# show the activity type, total hours and percentage share as text labels inside the pie slices
fig.update_traces(
    textposition="inside",
    texttemplate="<b>%{label}</b><br>%{value:.1f} h<br>%{percent}",
    hovertemplate=(
        "<b>%{label}</b><br>"
        "Study time: %{value:.1f} hours<br>"
        "Share: %{percent}<extra></extra>"
    ),
    rotation=60,
    domain=dict(
        x=[0.0, 0.78],
        y=[0.08, 1.0]
    )
)

# customize the layout of the figure for better aesthetics and readability
fig.update_layout(
    width=1000,
    height=650,
    title_font=dict(size=25),
    title=dict(
        text=(
            "Anatomy of a Study Routine<br>"
            "<sup>Total study time per activity type</sup>"
        ),
        x=0.07,
        xanchor="left"
    ),
    margin=dict(l=60, r=80, t=130, b=80),
    legend_title_text="Activity type",
    legend=dict(
        x=0.92,
        y=0.95,
        xanchor="right",
        yanchor="top"
    ),
    annotations=[
        dict(
            text="<b>Figure 2.</b> This chart shows how my total study time is divided between different types "
                 "of activities: it gives a quick overview<br>of the composition of my routine "
                 "and whether it is multi-faceted or or more one-sided.",
            x=0.03,
            y=-0.1,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=12)
        )
    ]
)

# save the figure as a static high-resolution image to the charts directory
fig.write_image(
    CHARTS_DIR / "total_study_time_per_activity.jpg",
    width=1000,
    height=650,
    scale=2
)

# display the figure in the notebook
fig.show()

The chart makes one thing painfully clear: my study time is mostly made of **programming**, with the other activities politely trying to exist around it.

- **Programming** dominates the chart with **102.5 hours**, which is both expected and slightly concerning. Apparently, my main study method is opening a code editor, breaking something, fixing it, breaking it again in a more sophisticated way, and calling that "progress". Still, it makes sense: most of my projects required writing scripts, manipulating data, debugging, and rebuilding things several times before they became usable.

- **Reading** comes second with **38.5 hours**, proving that I did, in fact, occasionally remember that university also involves reading. This includes papers, documentation, guidelines, and course materials: all the things I need before confidently misunderstanding a problem and then solving it with code.

- **Writing** is third with **28.25 hours**. This is the part where I try to transform the chaos of what I did into something that sounds intentional. It is less present than programming, but still essential, because unfortunately "the script works on my machine" is not usually accepted as a final exam report.

- **Reviewing** accounts for **14.5 hours**, which feels about right. In this log, reviewing does not mainly mean revising for exams, but checking other people's code, text, or task lists. It is a smaller part of the total workload, but it is still important because it usually happens when a project needs coordination, feedback or a final quality check before moving forward.

- **Brainstorming** is the smallest activity with **11 hours**. It mostly appears in group projects, when we need to understand the problem together, compare ideas, decide what direction to take, and turn confusion into something that looks at least vaguely like a plan. It takes less time than programming or writing, but it often shapes what happens afterwards.

Overall, the chart shows that my study routine during this period was very practical and project-based, which makes sense because I usually prefer tackling projects from the programming side rather than by writing long texts or reading a lot of papers. The useful lesson is that programming-heavy work should be planned with extra time, because it never takes "just one hour": that sentence is a lie, and I keep believing it.

---

### Visualization 3: Productivity by time of day and location

After looking at how my study time is divided by topic and activity, as a final step I want to understand something more practical: **where and when I seem to work better**. This visualization combines three parts of the log that are strongly connected to my everyday routine: `time_of_day`, `location`, and `productivity_score`. The idea is to check whether some combinations are more productive than others, for example if I tend to work better at home in the evening, in the library in the morning, or in some other very specific study habitat.

To visualize this, I will use a **bubble chart**. Each bubble represents a combination of `time_of_day` and `location`; the color shows the average `productivity_score`, while the size of the bubble represents the total `duration_hours` recorded for that combination. This is useful because it does not only show where productivity is higher, but also whether that result is based on a meaningful amount of study time or just on a tiny, suspiciously lucky session.

In [14]:
# group data by time of day and location, then calculate average productivity score, total hours and session count for each group
productivity_time_location = (
    df.groupby(["time_of_day", "location"], as_index=False)
    .agg(
        avg_productivity=("productivity_score", "mean"),
        total_hours=("duration_hours", "sum"),
        sessions=("productivity_score", "count")
    )
)

# compute the minimum and maximum average productivity values across all groups for normalization
group_min = productivity_time_location["avg_productivity"].min()
group_max = productivity_time_location["avg_productivity"].max()

# normalize the average productivity values to a 0-1 range for better color scaling in the heatmap
if group_max == group_min:
    productivity_time_location["avg_productivity_normalized"] = 1
else:
    productivity_time_location["avg_productivity_normalized"] = (
        (productivity_time_location["avg_productivity"] - group_min)
        / (group_max - group_min)
    )

# display the new dataframe for a quick check of the results
display(productivity_time_location)

,time_of_day,location,avg_productivity,total_hours,sessions,avg_productivity_normalized
0,morning,Bigiavi,7.700000,39.75,20,0.892857
1,morning,Home,7.000000,4.75,4,0.642857
2,morning,Palazzo Paleotti,7.222222,16.50,9,0.722222
3,morning,Zamboni 32,7.500000,18.50,8,0.821429
4,morning,Zamboni 34,8.000000,10.00,6,1.000000
5,morning,Zamboni 36,6.750000,6.25,4,0.553571
6,afternoon,Bigiavi,5.818182,21.50,11,0.220779
7,afternoon,Home,5.800000,14.00,5,0.214286
8,afternoon,Zamboni 34,5.600000,7.75,5,0.142857
9,afternoon,Zamboni 36,5.200000,8.50,5,0.000000


Since the average productivity values are quite close to each other, I normalized the grouped results after calculating the average productivity for each combination of time of day and location: this stretches the least productive combination to 0 and the most productive one to 1, making the differences easier to see in the color scale. The original average productivity score is still preserved for interpretation.

In [15]:
# create a bubble chart to show the relationship between time of day, location and productivity, with bubble size representing total hours and color representing normalized productivity
fig = px.scatter(
    productivity_time_location,
    x="time_of_day",
    y="location",
    size="total_hours",
    color="avg_productivity_normalized",
    hover_data={
        "avg_productivity": ":.2f",
        "avg_productivity_normalized": ":.2f",
        "total_hours": ":.2f",
        "sessions": True
    },
    title=(
        "The Study Sweet Spots<br>"
        "<sup>Productivity by time of day and location</sup>"
    ),
    labels={
        "time_of_day": "Time of day",
        "location": "Location",
        "avg_productivity_normalized": "Normalized productivity",
        "total_hours": "Total study hours",
        "sessions": "Number of sessions"
    },
    category_orders={
        "location": ["Home", "Palazzo Paleotti", "Bigiavi", "Zamboni 32", "Zamboni 34", "Zamboni 36"],
        "time_of_day": ["morning", "afternoon", "evening"]
    },
    size_max=55
)

# customize the layout of the figure for better aesthetics and readability, repositioning the legend and colorbar to the right of the plot area, and adding annotations to explain the chart elements
fig.update_layout(
    width=1100,
    height=750,
    margin=dict(l=120, r=240, t=110, b=160),
    title_font=dict(size=25),
    title=dict(
        text=(
            "The Coordinates of Focus<br>"
            "<sup>Productivity by time of day and location</sup>"
        ),
        x=0.125,
        xanchor="left"
    ),
    annotations=[
        dict(
            text=(
                "<b>Figure 3.</b> This chart shows how my productivity changes across different "
                "times of day and locations: each bubble<br>represents a specific combination of "
                "time and place, its size shows total study hours, and its color shows the "
                "productivity<br>score normalized across all combinations."
            ),
            x=0,
            y=-0.28,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="left",
            font=dict(size=12)
        ),
        dict(
            text="Normalized<br>productivity",
            x=1.20,
            y=0.93,
            xref="paper",
            yref="paper",
            showarrow=False,
            align="center",
            font=dict(size=14)
        )
    ],
    coloraxis_colorbar=dict(
        title=None,
        x=1.11,
        xanchor="left",
        y=0.43,
        yanchor="middle",
        len=0.75,
        thickness=32
    )
)

# update x and y axis titles with standoff for better spacing from the axis labels
fig.update_xaxes(
    title_text="Time of day",
    title_standoff=22
)

fig.update_yaxes(
    title_text="Location",
    title_standoff=22
)

# save the figure as a static high-resolution image to the charts directory
fig.write_image(
    CHARTS_DIR / "productivity_by_time_of_day_and_location.jpg",
    width=1100,
    height=750,
    scale=2
)

# save the figure as an interactive HTML file to the website directory for embedding
fig.write_html(
    WEBSITE_DIR / "chart.html",
    include_plotlyjs="cdn",
    full_html=True
)

# display the figure in the notebook
fig.show()

The chart has a very clear take-home message: **morning is carrying my productivity on its back**, while afternoon is where good intentions go to quietly suffer.

The strongest combinations are all in the **morning**. Location **34 in the morning** has the highest average productivity score, **8.0**, although it is based on only **10 hours** and **6 sessions**, so I should treat it as promising rather than absolute truth. **Bigiavi in the morning** is even more convincing: it has a high average productivity score, **7.7**, but also much more data behind it, with **39.75 hours** and **20 sessions**. This makes it probably the most reliable “sweet spot” in the chart: not just a lucky bubble, but a repeated pattern.

The afternoon, on the other hand, is consistently weak. Locations **34**, **36**, **Bigiavi**, and **home** all drop to an average productivity score around **5.2–5.8**. This is useful because it confirms that the problem is probably not just the place, but the time of day itself. Apparently, after lunch my brain enters a protected natural reserve where complex thoughts are not allowed.

The evening is mixed. **Bigiavi in the evening** has a lot of recorded time, **41.75 hours**, but only an average productivity score of **6.1**. This means I spend many evening hours there, but they are not necessarily my best hours. It may still be useful for lighter or mechanical work, but probably not for the tasks that require maximum focus.

The practical decision is clear: I should reserve **morning sessions**, especially at **Bigiavi** or location **34**, for the most demanding work: programming, writing important sections, or solving difficult project problems. Afternoons should be used more cautiously, either for lighter tasks, admin work, reviewing, or shorter sessions with clearer goals. Evenings can still be productive, but they should not become the place where I dump all the work I failed to do earlier, because the chart suggests that this strategy is not exactly a masterpiece of self-management.